# 第13章 テキストデータを用いた機械学習

『Python機械学習スタートブック』のコードをGoogle Colabで実行するためのノートブックです。
コードは書籍のリスト番号順に並んでいます。上から順に実行してください。

- 解説（Web教材）: https://ml.kano.ac/chapters/13/
- 演習の解答: https://ml.kano.ac/solutions/13/

## 文書分類

### データの準備

**リスト 13.1**　映画レビューデータの読み込みと確認

In [ ]:
import pandas as pd
from urllib.request import urlretrieve

# データのダウンロードと読み込み
url = "https://ml.kano.ac/chapters/data/movie_reviews.csv"
urlretrieve(url, "movie_reviews.csv")
df = pd.read_csv("movie_reviews.csv")

# 長文のテキストは30文字で省略して表示
pd.set_option("display.max_colwidth", 30)
print(df.shape)

# 肯定的・否定的なレビューを3件ずつ表示
print(df.groupby("label").head(3))

**リスト 13.2**　レビューのラベル分布の確認

In [ ]:
print(df["label"].value_counts())

### ナイーブベイズによる分類

**リスト 13.3**　TF-IDFとナイーブベイズによる文書分類

In [ ]:
import MeCab
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

# MeCabトークナイザの定義
tagger = MeCab.Tagger("-Owakati")

# MeCabで分かち書きを行い、単語のリストを返す
def tokenize(text):
    result = tagger.parse(text)
    return result.strip().split()

# テキストとラベルの取得
texts = df["text"].values
labels = df["label"].values

# TF-IDFベクトル化（MeCabトークナイザを指定）
vectorizer = TfidfVectorizer(tokenizer=tokenize, token_pattern=None)
X = vectorizer.fit_transform(texts)

# 訓練データとテストデータに分割
X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=42
)

# ナイーブベイズで学習
model_nb = MultinomialNB()
model_nb.fit(X_train, y_train)

# テストデータで評価
y_pred = model_nb.predict(X_test)
print("【ナイーブベイズ（TF-IDF）】")
print(classification_report(y_test, y_pred))

### SVMによる分類

**リスト 13.4**　線形SVMによる文書分類

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

# SVMで学習
model_svm = LinearSVC(max_iter=1000)
model_svm.fit(X_train, y_train)

# テストデータで評価
y_pred_svm = model_svm.predict(X_test)
print("【SVM（TF-IDF）】")
print(classification_report(y_test, y_pred_svm))

### 新しいテキストの予測

**リスト 13.5**　新しいレビューのカテゴリ予測

In [ ]:
# 新しいレビューを用意
new_reviews = [
    "最高の映画でした！感動して涙が出ました。",
    "つまらない映画でした。お金の無駄です。",
    "普通の映画でした。可もなく不可もなく。"
]

# TF-IDFベクトルに変換（学習時と同じvectorizerを使用）
X_new = vectorizer.transform(new_reviews)

# SVMで予測
predictions = model_svm.predict(X_new)

for review, pred in zip(new_reviews, predictions):
    print(f"レビュー: {review}")
    print(f"  予測: {pred}\n")

## 感情分析

**リスト 13.6**　学習済みモデルによる商品レビューの感情判定

In [ ]:
# 前節の学習済みモデルで商品レビューの感情を判定
product_reviews = [
    "この商品はとても使いやすくて満足しています",
    "期待外れの商品でした。残念です",
]
X_product = vectorizer.transform(product_reviews)
predictions = model_svm.predict(X_product)

for review, pred in zip(product_reviews, predictions):
    print(f"{review} → {pred}")

## Word2Vec、Doc2Vec

### gensimを用いたWord2Vecの学習

**リスト 13.7**　gensimのインストール

In [ ]:
# Google Colabでの実行
!pip install gensim

**リスト 13.8**　Word2Vecモデルの学習

In [ ]:
import MeCab
from gensim.models import Word2Vec

# サンプルの文書コーパス
corpus = [
    "機械学習はデータからパターンを学習する技術です",
    "深層学習はニューラルネットワークを用いた機械学習の手法です",
    "自然言語処理はテキストデータを分析する技術です",
    "画像認識は写真や動画から物体を検出する技術です",
    "機械学習を使ってテキストを分類することができます",
    "自然言語処理は人工知能の重要な分野です",
    "深層学習により画像認識の精度が大幅に向上しました",
    "テキストデータの分析には自然言語処理の技術が必要です",
]

# 内容語（名詞・動詞・形容詞）のみを抽出するトークナイザ
tagger = MeCab.Tagger()

def tokenize_content(text):
    target_pos = ["名詞", "動詞", "形容詞"]
    words = []
    node = tagger.parseToNode(text)
    while node:
        if node.surface != "" and \
           node.feature.split(",")[0] in target_pos:
            words.append(node.surface)
        node = node.next
    return words

tokenized_corpus = [tokenize_content(doc) for doc in corpus]

# Word2Vecモデルの学習
model_w2v = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=20,    # ベクトルの次元数
    window=5,          # 文脈の窓幅
    min_count=1,       # 最低出現回数
    epochs=500,        # 学習回数
    sg=1,              # 1: Skip-gram, 0: CBOW
    seed=42,           # 乱数シードの固定
    workers=1,         # 再現性のため単一スレッドで学習
)

# 学習結果の確認
print(f"語彙数: {len(model_w2v.wv)}")
print(f"「学習」のベクトル（先頭5要素）: {model_w2v.wv['学習'][:5]}")

**リスト 13.9**　「言語」に類似する単語の検索

In [ ]:
# 「言語」に類似する単語を検索
similar_words = model_w2v.wv.most_similar("言語", topn=5)
print("「言語」に類似する単語:")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

### Doc2Vecの概要と文書ベクトルの学習

**リスト 13.10**　Doc2Vecモデルの学習

In [ ]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

# TaggedDocumentの作成（文書IDとトークン化されたテキスト）
tagged_docs = [
    TaggedDocument(words=tokens, tags=[f"doc_{i}"])
    for i, tokens in enumerate(tokenized_corpus)
]

# Doc2Vecモデルの学習
model_d2v = Doc2Vec(
    documents=tagged_docs,
    vector_size=20,    # ベクトルの次元数
    window=5,          # 文脈の窓幅
    min_count=1,       # 最低出現回数
    epochs=500,        # 学習回数
    dm=1,              # 1: PV-DM（分散記憶モデル）, 0: PV-DBOW
    seed=42,           # 乱数シードの固定
    workers=1,         # 再現性のため単一スレッドで学習
)

# 文書ベクトルの確認
print(f"文書数: {len(tagged_docs)}")
print(f"doc_0 のベクトル（先頭5要素）: {model_d2v.dv['doc_0'][:5]}")

**リスト 13.11**　文書間の類似度の確認

In [ ]:
# 文書間の類似度
similar_docs = model_d2v.dv.most_similar("doc_0", topn=3)
print(f"元の文書: {corpus[0]}\n")
print("類似する文書:")
for doc_id, score in similar_docs:
    idx = int(doc_id.split("_")[1])
    print(f"  {doc_id} (類似度: {score:.4f}): {corpus[idx]}")

**リスト 13.12**　新しい文書のベクトル推論と類似文書検索

In [ ]:
# 新しい文書のベクトルを推論
new_doc = "データ分析に機械学習を活用する"
new_tokens = tokenize_content(new_doc)
new_vector = model_d2v.infer_vector(new_tokens)

# 類似文書の検索
similar = model_d2v.dv.most_similar([new_vector], topn=3)
print(f"クエリ: {new_doc}\n")
print("類似する文書:")
for doc_id, score in similar:
    idx = int(doc_id.split("_")[1])
    print(f"  {doc_id} (類似度: {score:.4f}): {corpus[idx]}")

## 文脈を考慮したEmbedding

### Sentence Transformers

**リスト 13.13**　Sentence Transformersのインストール

In [ ]:
# Google Colabでの実行
!pip install sentence-transformers

**リスト 13.14**　文のEmbeddingへの変換

In [ ]:
from sentence_transformers import SentenceTransformer

# モデルの読み込み
model_st = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# テキストをEmbeddingに変換
sentences = [
    "この映画は素晴らしかった",
    "この映画は最高だった",
    "今日の天気は晴れです"
]

embeddings = model_st.encode(sentences)

print(f"文の数: {len(embeddings)}")
print(f"各ベクトルの次元数: {embeddings.shape[1]}")
print(f"最初の文のベクトル（先頭5要素）: {embeddings[0][:5]}")

### Sentence Transformersによる文書分類

**リスト 13.15**　Embeddingを特徴量とした文書分類

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 映画レビューデータを使用
texts = df["text"].values.tolist()
labels = df["label"].values

# 前項で読み込んだモデルでEmbeddingを生成
embeddings = model_st.encode(texts)

print(f"Embeddingの形状: {embeddings.shape}")

# 訓練データとテストデータに分割
X_train, X_test, y_train, y_test = train_test_split(
    embeddings, labels, test_size=0.2, random_state=42
)

# ロジスティック回帰で学習
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train, y_train)

# テストデータで評価
y_pred = model_lr.predict(X_test)
print(classification_report(y_test, y_pred))

## 演習問題

まず、演習で使うデータを準備します。

In [ ]:
# 演習で使うデータのダウンロード
from urllib.request import urlretrieve

urlretrieve("https://ml.kano.ac/chapters/data/emails.csv", "emails.csv")

### 演習 13-1: TF-IDF と分類器の比較

映画レビューのテキストと感情ラベル（positive / negative）が各 100 件・計 200 件入った [`movie_reviews.csv`](data/movie_reviews.csv) を使って、**TF-IDF ベクトル** に対する **ナイーブベイズ・SVM・ロジスティック回帰** の 3 つの分類器のテスト精度を比較してください。さらに、ロジスティック回帰が学習した係数から「ポジティブ寄り」「ネガティブ寄り」の重要語を確認します。

**タスク**

1. CSV を読み込み、`MeCab.Tagger("-Owakati")` で分かち書きするトークナイザ関数を定義する
2. `TfidfVectorizer(tokenizer=tokenize, token_pattern=None)` で TF-IDF ベクトル化し、訓練 80%・テスト 20% に分割する（`random_state=42`）
3. `MultinomialNB()`・`LinearSVC(max_iter=1000)`・`LogisticRegression(max_iter=1000)` の 3 モデルで学習し、テスト正解率を比較する
4. ロジスティック回帰の `coef_` を使って、係数が **最大の 10 単語**（ポジティブ寄り）と **最小の 10 単語**（ネガティブ寄り）を表示し、どのような単語が判定の手がかりになっているか考察する

In [ ]:
import numpy as np
import pandas as pd
import MeCab
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. データ読み込み + トークナイザ関数の定義

# 2. TF-IDF ベクトル化 + 訓練・テストに分割

# 3. 3 モデルで学習・正解率比較

# 4. ロジスティック回帰の係数から上位 10 / 下位 10 単語を表示・考察

[解答例を見る](https://ml.kano.ac/solutions/13/#solution-13-1)

### 演習 13-2: ベクトル化の設定変更

`TfidfVectorizer` に `ngram_range=(1, 2)` を指定して単語 2 つの並び（バイグラム）まで特徴量に含めた場合、SVM の分類精度がどう変わるか確認してください。

**タスク**

1. 演習 13-1 と同じデータ・トークナイザを使い、`ngram_range=(1, 2)` を指定した `TfidfVectorizer` でベクトル化する
2. 同じ条件（`test_size=0.2, random_state=42`）で分割し、`LinearSVC(max_iter=1000)` で学習する
3. 語彙数とテスト正解率を unigram のみの場合と比較し、結果の理由を考察する

[解答例を見る](https://ml.kano.ac/solutions/13/#solution-13-2)

### 演習 13-3: 自作レビューの判定

自分で肯定的なレビューと否定的なレビューを 2 文ずつ作成し、演習 13-1 の学習済み SVM モデルで感情を判定してください。誤判定された文があれば、どのような単語が原因か考察してください。

**タスク**

1. 肯定的なレビュー 2 文・否定的なレビュー 2 文を自作する
2. 演習 13-1 の `vectorizer` で TF-IDF ベクトルに変換する（`vectorizer.transform()`）
3. 学習済み SVM モデルで判定し、結果を表示する
4. 誤判定された文があれば、原因となった単語を考察する

[解答例を見る](https://ml.kano.ac/solutions/13/#solution-13-3)

### 演習 13-4: 文の類似度と類似文検索

Sentence Transformers を使って、文の意味の近さを数値化してみましょう。前半では 3 つの文のコサイン類似度を計算し、後半では 1 つのクエリ文に対して候補文のリストの中から意味が最も近い文を探すプログラム（類似文検索）を作成します。

**タスク**

1. `SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")` で「この映画は面白かった」「この作品はとても楽しめた」「今日は雨が降っています」の 3 文を Embedding に変換する
2. 1 文目と 2 文目、1 文目と 3 文目のコサイン類似度（`util.cos_sim`）を計算し、意味の近い文のほうが高い類似度になることを確認する
3. クエリ文と候補文をまとめて Embedding に変換し、クエリと各候補の類似度を表示して、最も類似度が高い候補文を出力する

In [ ]:
from sentence_transformers import SentenceTransformer, util

# モデルの読み込み
model_st = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# 1-2. 3 文を Embedding に変換し、コサイン類似度を計算する

# 3. 類似文検索
query = "スマートフォンの新製品が発表された"
candidates = [
    "新しいスマホが登場して話題になっている",
    "今日の夕食はカレーライスにした",
    "台風が接近し、明日は大雨の予報だ",
    "最新の携帯電話が公開され注目を集めた",
]

[解答例を見る](https://ml.kano.ac/solutions/13/#solution-13-4)

### 演習 13-5: TF-IDF と Embedding の精度比較

課題 13 で使う [`emails.csv`](data/emails.csv)（ビジネス・プライベート・スパムの 3 クラス、各 100 件・計 300 件）を使って、TF-IDF と Sentence Transformers の Embedding でメール分類モデルを作り、テスト精度を比較してください。ベクトル化の方法だけを変え、分類器（ロジスティック回帰）と分割条件は揃えます。どちらが高精度か、なぜかも考えてみましょう。

**タスク**

1. CSV を読み込む（ラベルは `business`・`private`・`spam` の 3 種類。文字列のまま使ってよい）
2. 公平に比べるため、先に `train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])` で 1 回だけ訓練・テストに分割し、同じ行を両手法で使う
3. TF-IDF 版：`MeCab.Tagger("-Owakati")` で分かち書き → `TfidfVectorizer` → ロジスティック回帰で精度を求める
4. Embedding 版：`model_st.encode()` でベクトル化 → ロジスティック回帰で精度を求める
5. 2 つの精度を並べて表示し、どちらが高いか・なぜかを考察する

In [ ]:
import pandas as pd
import MeCab
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. データ読み込み（ラベルは文字列のまま使う）
df = pd.read_csv('emails.csv')

# 2. 先に 1 回だけ train_test_split で分割

# 3. TF-IDF + ロジスティック回帰で精度を求める

# 4. Sentence Transformers の Embedding + ロジスティック回帰で精度を求める

# 5. 2 つの精度を並べて表示・考察

[解答例を見る](https://ml.kano.ac/solutions/13/#solution-13-5)

### 発展課題: アンカー文による学習なし感情判定

学習をせずに、ポジティブ／ネガティブを代表する 2 つの「アンカー文」との類似度だけで、文の感情を判定してみましょう。判定したい文を 2 つのアンカーと比べ、近い方のラベルを付けます。

**タスク**

1. アンカー文（`pos_anchor`・`neg_anchor`）とテスト文をまとめて `model_st.encode()` でベクトル化する
2. `util.cos_sim` で各テスト文とアンカー文の類似度を計算する
3. 各テスト文について、pos アンカーとの類似度が neg アンカーより高ければ `positive`、そうでなければ `negative` と判定して表示する

In [ ]:
from sentence_transformers import SentenceTransformer, util

model_st = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# ポジ・ネガを代表するアンカー文
pos_anchor = "素晴らしい映画だった"
neg_anchor = "ひどい映画だった"

# 判定したい文
tests = [
    "期待以上の出来で本当に感動した",
    "時間の無駄だった。二度と見たくない",
    "話の展開が退屈で全く楽しめなかった",
    "映像が美しく、心から楽しめる作品だった",
]

# 1. アンカー文とテスト文をまとめてベクトル化する

# 2. コサイン類似度を計算する（util.cos_sim）

# 3. 各テスト文について pos/neg アンカーとの類似度を比べ、近い方のラベルを表示する

[解答例を見る](https://ml.kano.ac/solutions/13/#solution-13-adv-1)